# IOAI — 2024 First Stage Dependency Parsing (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
if not os.path.exists('data/train.conll'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2024-first-stage-dependency-parsing/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 의존 구문 분석 — 구조 프로브 모범답안 (Hewitt-Manning Structural Probe)

**아이디어**: 사전학습 언어모델 **HerBERT** 의 어절 임베딩 h 에 학습 가능한 선형 변환 B 를 씌워
- **DistanceModel**: 두 어절 거리 = `||B(h_i - h_j)||²` 를 **트리 거리**(간선 수)에 회귀,
- **DepthModel**: 각 어절 깊이 = `||B h_i||²` 를 **루트로부터의 거리**에 회귀.

학습 후, 예측 거리 행렬로 **최소신장트리(MST)** 를 만들어 무방향 의존 트리를, 예측 깊이 최소 어절을
**루트**로 골라 방향을 부여한다. HerBERT 는 **동결**(임베딩만 사용), 학습되는 건 선형 프로브 B 뿐이다.

**점수(valid 200문장, 실측)**: UUAS ≈ **58.6%** · root ≈ 23% → **≈ 0.25 / 2.0**
(베이스라인 선형체인 ≈ 0.14). *정직한 한계*: 영어 PTB(4만 문장)는 UUAS 80%지만, 여기는 폴란드어
**1000문장 + 선형 프로브**라 절대 UUAS 가 낮다. 그래도 구조 프로브의 핵심 기법을 그대로 재현한다.
채점식이 UUAS·root 각각 0.5 미만이면 0점 처리하므로 점수의 대부분은 UUAS(≈0.59)에서 나온다.


In [ ]:
# 환경 + 데이터 (Colab: HerBERT/데이터 자동 준비, ~수 분·GPU 권장)
import os, urllib.request, zipfile
if not os.path.exists("data/train.conll"):
    url = "https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2024-first-stage-dependency-parsing/data.zip"
    urllib.request.urlretrieve(url, "d.zip"); zipfile.ZipFile("d.zip").extractall("data")

import collections, csv, numpy as np, torch, torch.nn as nn
from scipy.sparse.csgraph import minimum_spanning_tree
from transformers import AutoModel, AutoTokenizer
dev = "cuda" if torch.cuda.is_available() else "cpu"
LAYER, K, EPOCHS = 8, 128, 300           # HerBERT 은닉층 8, 프로브 랭크 128, 에폭 300

def read_conll(fp):
    sents, ws, hs = [], [], []
    for line in open(fp, encoding="utf-8"):
        sp = line.strip().split("\t")
        if len(sp) >= 7: ws.append(sp[1]); hs.append(int(sp[6]))
        elif ws: sents.append((ws, hs)); ws, hs = [], []
    if ws: sents.append((ws, hs))
    return sents

train = read_conll("data/train.conll"); valid = read_conll("data/valid.conll")
print("train", len(train), "valid", len(valid), "| dev", dev)


In [ ]:
# HerBERT 로드(동결) + 어절 임베딩(서브워드 평균풀링)
tokenizer = AutoTokenizer.from_pretrained("allegro/herbert-base-cased")
bert = AutoModel.from_pretrained("allegro/herbert-base-cased").to(dev).eval()

@torch.no_grad()
def word_embeddings(words):
    enc = tokenizer(words, is_split_into_words=True, return_tensors="pt", truncation=True, max_length=256)
    wid = enc.word_ids()
    out = bert(**{k: v.to(dev) for k, v in enc.items()}, output_hidden_states=True)
    hs = out.hidden_states[LAYER][0]                    # (T, 768)
    buck = collections.defaultdict(list)
    for t, w in enumerate(wid):
        if w is not None: buck[w].append(hs[t])
    return torch.stack([torch.stack(buck[i]).mean(0) for i in range(len(words))]).cpu()  # (L,768)

def tree_distances(heads):                              # 골드 트리의 어절쌍 거리(BFS)
    n = len(heads); adj = [[] for _ in range(n)]
    for i, h in enumerate(heads):
        if h != 0: adj[i].append(h-1); adj[h-1].append(i)
    D = np.zeros((n, n))
    for s in range(n):
        d = [-1]*n; d[s] = 0; q = collections.deque([s])
        while q:
            u = q.popleft()
            for v in adj[u]:
                if d[v] < 0: d[v] = d[u]+1; q.append(v)
        D[s] = d
    return D

def gold_root(heads): return heads.index(0)
print("임베딩 계산 중...")
tr_emb = [word_embeddings(w) for w, _ in train]
va_emb = [word_embeddings(w) for w, _ in valid]
tr_dist = [tree_distances(h) for _, h in train]
print("완료")


In [ ]:
class DistanceModel(nn.Module):
    """||B(h_i - h_j)||² 로 어절쌍 트리 거리를 예측."""
    def __init__(self, d=768, k=K):
        super().__init__(); self.B = nn.Linear(d, k, bias=False)
    def forward(self, e):                 # e:(L,768) -> (L,L)
        t = self.B(e); df = t.unsqueeze(0) - t.unsqueeze(1)
        return (df ** 2).sum(-1)

class DepthModel(nn.Module):
    """||B h_i||² 로 루트로부터의 깊이를 예측."""
    def __init__(self, d=768, k=K):
        super().__init__(); self.B = nn.Linear(d, k, bias=False)
    def forward(self, e):                 # e:(L,768) -> (L,)
        t = self.B(e); return (t ** 2).sum(-1)

def train_probe(kind):
    model = (DistanceModel() if kind == "dist" else DepthModel()).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for ep in range(EPOCHS):
        for i in np.random.permutation(len(tr_emb)):
            e = tr_emb[i].to(dev); D = torch.tensor(tr_dist[i], dtype=torch.float32, device=dev)
            L = e.shape[0]
            if kind == "dist":
                loss = (model(e) - D).abs().sum() / (L * L)             # 어절쌍 L1, 길이²로 정규화
            else:
                loss = (model(e) - D[gold_root(train[i][1])]).abs().sum() / L
            opt.zero_grad(); loss.backward(); opt.step()
    return model.eval()

print("거리 프로브 학습..."); distance_model = train_probe("dist")
print("깊이 프로브 학습..."); depth_model = train_probe("depth")


In [ ]:
# 예측 거리 -> MST 로 무방향 트리, 예측 깊이 -> 루트, 방향 부여
@torch.no_grad()
def parse_sentence(words, distance_model, depth_model):
    e = word_embeddings(words).to(dev); L = e.shape[0]
    if L == 1: return [0]
    pd = distance_model(e).cpu().numpy()
    mst = minimum_spanning_tree(pd).toarray(); mst = mst + mst.T
    root = int(depth_model(e).argmin().item())
    adj = [[] for _ in range(L)]
    for a in range(L):
        for b in range(L):
            if mst[a, b] > 0: adj[a].append(b)
    heads = [0] * L; dq = collections.deque([(root, -1)]); seen = {root}
    while dq:
        v, par = dq.popleft(); heads[v] = par + 1
        for u in adj[v]:
            if u not in seen: seen.add(u); dq.append((u, v))
    return heads

def edges(h): return set((min(i, x-1), max(i, x-1)) for i, x in enumerate(h) if x != 0)
uu = rc = 0; rows = []
for sid, (words, heads) in enumerate(valid):
    ph = parse_sentence(words, distance_model, depth_model)
    ge = edges(heads); uu += len(ge & edges(ph)) / len(ge); rc += int(ph.index(0) == gold_root(heads))
    for tid, h in enumerate(ph): rows.append([sid, tid + 1, h])
print(f"UUAS {uu/len(valid)*100:.1f}%  root {rc/len(valid)*100:.1f}%")
with open("submission.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["sent_id", "token_id", "head"]); w.writerows(rows)
print("submission.csv 저장:", len(rows), "행")


### 정리
- **DistanceModel / DepthModel** = Hewitt-Manning 구조 프로브. HerBERT 동결, 선형 프로브만 학습.
- 예측 거리→**MST**, 예측 깊이 최소→**루트**. UUAS ≈ 0.59, root ≈ 0.23 → ≈ 0.25/2.0 (베이스라인 0.14).
- **더 끌어올리려면**: 더 큰 사전학습모델·다층 임베딩 결합·랭크/에폭 확대, root 는 별도 지도학습 헤드.
  (여기서는 원문제 스캐폴드의 두 프로브 구조를 충실히 재현하는 데 초점.)


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)